# Typecheck `PermanentProofs.lean` in Google Colab

This notebook sets up **Lean 4 (`v4.36.0`)** and **Mathlib4** in Google Colab and typechecks `lean/PermanentProofs.lean`.

Steps:
1. Install `elan` (Lean version manager) and Lean `v4.36.0`.
2. Create a minimal Lake project with `mathlib` (`v4.36.0`) and fetch precompiled `.olean` files via `lake exe cache get`.
3. Copy `lean/PermanentProofs.lean` into the Lake project (or upload it if running standalone).
4. Run `lake env lean PermanentProofs.lean` to typecheck all theorems.

In [ ]:
%%bash
set -euo pipefail

# 1. Install elan and Lean v4.36.0
if ! command -v elan >/dev/null 2>&1; then
  curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.36.0
fi

export PATH="$HOME/.elan/bin:$PATH"
elan toolchain install leanprover/lean4:v4.36.0
elan default leanprover/lean4:v4.36.0

lean --version
lake --version

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.elan/bin:$PATH"

# 2. Create minimal Lake project with Mathlib4 (v4.36.0) and fetch precompiled .olean cache
mkdir -p perm_proofs
cd perm_proofs

cat <<'EOF' > lean-toolchain
leanprover/lean4:v4.36.0
EOF

cat <<'EOF' > lakefile.toml
name = "perm_proofs"
version = "0.1.0"
defaultTargets = ["PermanentProofs"]

[[require]]
name = "mathlib"
git = "https://github.com/leanprover-community/mathlib4.git"
rev = "v4.36.0"

[[lean_lib]]
name = "PermanentProofs"
EOF

lake update
lake exe cache get

In [ ]:
# 3. Copy lean/PermanentProofs.lean into the Lake project (or upload in Colab)
import os
import shutil

dest = os.path.join("perm_proofs", "PermanentProofs.lean")
candidates = [
    os.path.join("lean", "PermanentProofs.lean"),
    os.path.join("..", "lean", "PermanentProofs.lean"),
    "PermanentProofs.lean",
]

src = next((p for p in candidates if os.path.isfile(p)), None)
if src is not None:
    shutil.copy2(src, dest)
    print(f"Copied {src} -> {dest}")
else:
    print("lean/PermanentProofs.lean not found locally. Please upload PermanentProofs.lean:")
    from google.colab import files
    uploaded = files.upload()
    fname = next(iter(uploaded))
    shutil.move(fname, dest)
    print(f"Saved uploaded file -> {dest}")

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.elan/bin:$PATH"

# 4. Typecheck all theorems in PermanentProofs.lean
cd perm_proofs
lake env lean PermanentProofs.lean
echo "SUCCESS: PermanentProofs.lean typechecked cleanly!"